# Generative AI and Foundation Models

So far our models labeled things (spam or not, which digit) or predicted a number. **Generative AI** creates new content: text, images, code, audio. A **foundation model** is a very large model trained on broad data and then adapted to many tasks. A **large language model (LLM)** is a foundation model for text. Claude, ChatGPT, and Gemini are LLMs.

**Key terms**
- Token: the piece of text a model reads and writes (a word, part of a word, or a character)
- Embedding: a list of numbers that represents the meaning of a word, sentence, or image
- Prompt: the instructions and context you give the model
- Temperature: a setting that controls how random the model's choices are
- Multimodal: a model that handles more than one kind of data (text, images, audio)
- Hallucination: confident, fluent output that is false or made up

**Today is a short one.** Real LLMs have billions of parameters, so we build tiny versions of the core ideas that run in seconds. Same format as before: a demo, then a short Your Turn with a blank (set to None) to fill in. Prompting is tested in a chatbot for the assignment.

**Roadmap:** 1. Tokens  2. Embeddings  3. Generating text one token at a time  4. Prompting and hallucinations

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nltk
from collections import Counter
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

nltk.download('movie_reviews', quiet=True)
nltk.download('brown', quiet=True)
from nltk.corpus import movie_reviews, brown

def clean(words):
    return [w.lower() for w in words if w.isalpha()]

demo_words = clean(movie_reviews.words())   # demo text: movie reviews
your_words = clean(brown.words())           # your text: a mix of news, fiction, and more

print('Demo text:', len(demo_words), 'words.   Your text:', len(your_words), 'words.')

---
## Step 1 - Tokens

A model does not read words. It reads **tokens**. A common method, **byte pair encoding (BPE)**, starts with single characters and repeatedly merges the most frequent neighboring pair into a new token. Common words end up as one token, and rare words get split into pieces.

Tokens matter because a model's memory (its **context window**) and its price are both measured in tokens. A rough rule: one token is about three quarters of a word.

Below is a tiny BPE trainer. The `_` marks the end of a word.

In [ ]:
def train_bpe(words, num_merges):
    vocab = Counter(tuple(w) + ('</w>',) for w in words)
    merges = []
    for _ in range(num_merges):
        pairs = Counter()
        for word, freq in vocab.items():
            for a, b in zip(word, word[1:]):
                pairs[(a, b)] += freq
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        new_vocab = Counter()
        for word, freq in vocab.items():
            new, i = [], 0
            while i < len(word):
                if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                    new.append(word[i] + word[i + 1]); i += 2
                else:
                    new.append(word[i]); i += 1
            new_vocab[tuple(new)] += freq
        vocab = new_vocab
    return merges

def tokenize(word, merges):
    tokens = list(word) + ['</w>']
    for a, b in merges:
        new, i = [], 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new.append(a + b); i += 2
            else:
                new.append(tokens[i]); i += 1
        tokens = new
    return tokens

def show_tokens(word, merges):
    return ' | '.join(tokenize(word, merges)).replace('</w>', '_')

merges_demo = train_bpe(demo_words[:30000], 300)   # takes a few seconds

rows = {w: {f'{n} merges': show_tokens(w, merges_demo[:n]) for n in [10, 100, 300]}
        for w in ['movie', 'unforgettable', 'zzyzx']}
pd.set_option('display.max_colwidth', 80)
display(pd.DataFrame(rows).T)

With few merges everything is split into letters. With more merges, common words like "movie" become a single token, while rare or invented words stay in pieces. Real LLMs work the same way, with vocabularies of 100,000 or more tokens.

### Your Turn - Train your own tokenizer

TODO: set two values.
- `NUM_MERGES`: how many merges to learn (try 20, 100, 300)
- `TOKEN_WORD`: any lowercase word to tokenize (try a common word, a long word, and a made-up word)

In [ ]:
NUM_MERGES = None   # TODO: try 20, 100, 300
TOKEN_WORD = None   # TODO: a lowercase word, e.g. 'government' or 'blorptastic'
assert NUM_MERGES is not None and TOKEN_WORD is not None, 'Set NUM_MERGES and TOKEN_WORD above first, then re-run this cell.'

merges_yours = train_bpe(your_words[:30000], NUM_MERGES)   # takes a few seconds
tokens = tokenize(TOKEN_WORD.lower(), merges_yours)
print('Tokens:', show_tokens(TOKEN_WORD.lower(), merges_yours))
print('Number of tokens:', len(tokens), ' for a word with', len(TOKEN_WORD), 'letters')

**Your answers:**
1. How did the number of tokens change as you increased `NUM_MERGES`? Which kinds of words stayed split into pieces?
2. Why would an LLM prefer a vocabulary of common whole words and pieces over single letters or every possible word?

---
## Step 2 - Embeddings

Models need numbers, so each token is turned into an **embedding**: a list of numbers (a vector) where similar meanings get similar numbers. Embeddings are learned from context: words that appear in similar surroundings end up with similar vectors. We measure closeness with **cosine similarity** (1 means same direction, 0 means unrelated).

Embeddings power search, recommendations, and the first layer inside every LLM. Multimodal models learn them for images and audio too, so a photo and its caption land close together.

Below we build small embeddings from word co-occurrence counts, a simpler cousin of how word embeddings are really trained.

In [ ]:
def build_embeddings(words, vocab_size=3000, window=4, dim=50):
    counts = Counter(words)
    vocab = [w for w, _ in counts.most_common(vocab_size)]
    index = {w: i for i, w in enumerate(vocab)}
    ids = np.array([index.get(w, -1) for w in words])

    rows, cols = [], []
    for k in range(1, window + 1):
        a, b = ids[:-k], ids[k:]
        keep = (a >= 0) & (b >= 0)
        rows += [a[keep], b[keep]]
        cols += [b[keep], a[keep]]
    rows, cols = np.concatenate(rows), np.concatenate(cols)

    V = len(vocab)
    co = sparse.coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(V, V)).tocsr().tocoo()
    total = co.data.sum()
    row_sums = np.bincount(co.row, weights=co.data, minlength=V)
    col_sums = np.bincount(co.col, weights=co.data, minlength=V)
    pmi = np.log(co.data * total / (row_sums[co.row] * col_sums[co.col]))
    matrix = sparse.coo_matrix((np.maximum(pmi, 0), (co.row, co.col)), shape=(V, V)).tocsr()

    vectors = TruncatedSVD(n_components=dim, random_state=42).fit_transform(matrix)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
    return vocab, index, vectors

def nearest(word, vocab, index, vectors, n=6):
    sims = vectors @ vectors[index[word]]
    best = np.argsort(-sims)[1:n + 1]
    return [(vocab[i], round(float(sims[i]), 2)) for i in best]

def similarity(a, b, index, vectors):
    return round(float(vectors[index[a]] @ vectors[index[b]]), 2)

vocab_d, index_d, vectors_d = build_embeddings(demo_words)

for w in ['boring', 'actor', 'terrible']:
    print(f'Closest to "{w}":', nearest(w, vocab_d, index_d, vectors_d))

print()
print('similarity(terrible, awful):', similarity('terrible', 'awful', index_d, vectors_d))
print('similarity(terrible, actor):', similarity('terrible', 'actor', index_d, vectors_d))

Nobody told the computer that "boring" and "dull" mean almost the same thing. It learned that from the words they appear near. It is not perfect (some neighbors are just words that show up in the same sentences), but this is the core idea behind how models represent meaning.

### Your Turn - Explore your own embeddings

Your text is general-purpose (news, fiction, and more), so try everyday words.

TODO: set two values.
- `WORD`: a word to look up (try `'president'`, `'school'`, `'city'`, `'money'`)
- `COMPARE`: a second word to compare it with

In [ ]:
vocab_y, index_y, vectors_y = build_embeddings(your_words)

WORD = None      # TODO: a common word, e.g. 'president', 'school', 'city', 'money'
COMPARE = None   # TODO: a second common word to compare with WORD
assert WORD is not None and COMPARE is not None, 'Set WORD and COMPARE above first, then re-run this cell.'
assert WORD in index_y and COMPARE in index_y, 'Both words must be common words in the text. Try another word.'

print(f'Closest to "{WORD}":', nearest(WORD, vocab_y, index_y, vectors_y))
print(f'Similarity between "{WORD}" and "{COMPARE}":', similarity(WORD, COMPARE, index_y, vectors_y))

**Your answers:**
1. Which neighbors made sense and which were surprising? Why might a surprising word appear near yours?
2. Compare a related pair with an unrelated pair by changing `COMPARE`. What similarity scores did you get?

---
## Step 3 - Generating Text One Token at a Time

This is the core of an LLM: **predict the next token, pick one, add it to the text, and repeat.** Real LLMs use huge neural networks that look at the whole prompt. Our toy looks only at the previous word, learned by counting which word follows which (a bigram model).

**Temperature** controls how the next word is picked:
- Low temperature: almost always the most likely word (predictable, can get repetitive)
- High temperature: more random (more varied, less coherent)

Notice what is missing: nothing in this process checks whether the text is true. The model only picks likely next words.

In [ ]:
def build_bigram(words):
    model = {}
    for a, b in zip(words, words[1:]):
        model.setdefault(a, Counter())[b] += 1
    return model

def generate(model, start, length=15, temperature=1.0, seed=42):
    rng = np.random.default_rng(seed)
    text = [start]
    for _ in range(length):
        following = model.get(text[-1])
        if not following:
            break
        options = list(following)
        probs = np.array([following[w] for w in options], dtype=float) ** (1 / temperature)
        probs /= probs.sum()
        text.append(str(rng.choice(options, p=probs)))
    return ' '.join(text)

bigram_d = build_bigram(demo_words)

for temp in [0.2, 1.0, 3.0]:
    print(f'Temperature {temp}:')
    print('  ', generate(bigram_d, 'the', temperature=temp))

At low temperature the text loops on its most likely phrases. At high temperature it becomes word salad. In between it reads fluently but means nothing, which is a small-scale picture of a hallucination: fluent text with no check on truth.

### Your Turn - Generate from your text

TODO: set two values.
- `START`: the first word (try `'the'`, `'he'`, `'in'`, or any common word)
- `TEMPERATURE`: a number above 0 (try 0.2, 1.0, 3.0)

In [ ]:
bigram_y = build_bigram(your_words)

START = None         # TODO: a common word such as 'the', 'he', 'in'
TEMPERATURE = None   # TODO: a number above 0. Try 0.2, 1.0, 3.0.
assert START is not None and TEMPERATURE is not None and TEMPERATURE > 0, 'Set START and TEMPERATURE (above 0) first, then re-run this cell.'
assert START in bigram_y, 'That start word is not in the text. Try a more common word.'

print(generate(bigram_y, START, length=20, temperature=TEMPERATURE))

**Your answers:**
1. Compare a low and a high temperature. Describe the difference in the text.
2. The generated text often sounds plausible. Why does that make errors hard to spot?

---
## Step 4 - Prompting and Hallucinations

**Prompting** is how you steer a foundation model. Good prompts usually:
- State the task clearly and specifically
- Give context or a role ("You are a support agent for a bank")
- Include examples of the input and the output you want (**few-shot**), or none (**zero-shot**)
- Specify the output format (one word, a table, three bullet points)
- Get refined over several tries

**Hallucinations** happen because an LLM predicts plausible text, not verified facts. They are more likely with obscure topics, questions with false premises, and requests for specific details like quotes, numbers, or citations. Ways to reduce them: ask for sources and check them, give the model the source text to work from, lower the temperature, tell it to say when it does not know, and keep a person in the loop for high-stakes decisions (the "Should we use AI?" question from Week 6).

**Multimodal** models accept and produce more than text: they can describe a photo, generate an image from a description, or transcribe and summarize audio. The same prompting and hallucination rules apply.

Below is a helper that assembles a few-shot prompt.

In [ ]:
def build_prompt(task, examples, new_input):
    lines = [task, '']
    for example_input, example_output in examples:
        lines += [f'Input: {example_input}', f'Output: {example_output}', '']
    lines += [f'Input: {new_input}', 'Output:']
    return '\n'.join(lines)

def approx_tokens(text):
    return int(len(text.split()) / 0.75)   # rough rule: 1 token is about 3/4 of a word

demo_prompt = build_prompt(
    'Classify the review as positive or negative. Answer with one word.',
    [('I loved the acting', 'positive'), ('What a waste of time', 'negative')],
    'The plot was dull but the music was great',
)
print(demo_prompt)
print('\nApproximate tokens:', approx_tokens(demo_prompt))

### Your Turn - Build a prompt for a task of your choice

TODO: set three values.
- `TASK`: the instruction (for example, turn a customer email into a one-line summary, or label a support message as urgent or not urgent)
- `EXAMPLES`: a list of at least two (input, output) pairs, such as `[('input one', 'output one'), ('input two', 'output two')]`
- `NEW_INPUT`: a new input for the model to handle

In [ ]:
TASK = None        # TODO: what should the model do? One clear sentence.
EXAMPLES = None    # TODO: a list of at least 2 (input, output) pairs
NEW_INPUT = None   # TODO: a new input for the model
assert TASK is not None and EXAMPLES is not None and NEW_INPUT is not None, 'Set TASK, EXAMPLES and NEW_INPUT above first, then re-run this cell.'
assert len(EXAMPLES) >= 2, 'Give at least 2 examples.'

my_prompt = build_prompt(TASK, EXAMPLES, NEW_INPUT)
print(my_prompt)
print('\nApproximate tokens:', approx_tokens(my_prompt))

**Your answers:**
1. Paste your prompt into a chatbot (Claude, ChatGPT, or Gemini). Did it follow your format? What did your examples teach it?
2. Name one way this task could go wrong with a hallucination, and what you would check.

---
# Assignment - Prompt It and Check It

Short assignment, two tasks. Run the notebook from the top first. Write your own explanations in the answer cells.

## Task 1 - Few-shot versus zero-shot

You built a few-shot prompt (with examples) in Step 4. Now write a zero-shot version of the same task (instructions only, no examples), and compare.

TODO: set `ZERO_SHOT_TASK` to your instructions. Be specific, since there are no examples to show the format.

In [ ]:
ZERO_SHOT_TASK = None   # TODO: the same task, written as detailed instructions with no examples
assert ZERO_SHOT_TASK is not None, 'Set ZERO_SHOT_TASK above first, then re-run this cell.'

zero_shot_prompt = build_prompt(ZERO_SHOT_TASK, [], NEW_INPUT)
print(zero_shot_prompt)
print('\nApproximate tokens, zero-shot:', approx_tokens(zero_shot_prompt))
print('Approximate tokens, few-shot: ', approx_tokens(my_prompt))

Now paste each prompt into a chatbot (a new chat each time) and compare the answers.

**Task 1 answer.** Did the examples change the format or quality of the answer? Which prompt would you use in a real product, and why? Consider both quality and token cost.

**Your answer:**

## Task 2 - Hallucination hunt

Ask a chatbot something it is likely to get wrong. Ideas: a question about a book, person, or company that does not exist, a question with a false premise, or a request for three academic sources on a narrow topic. Check its answer against reliable sources.

**Task 2 answer.** In 4 to 6 sentences: what did you ask, what did it answer, what was wrong or right, how does the next-word generator in Step 3 help explain what happened, and what one habit would you use to reduce this risk at work?

**Your answer:**